# nb_gold_dim_date

Build contiguous calendar dimension **`gold.dim_date`** for Direct Lake (physical Delta table — no DAX calculated date table).

**Attach lakehouse:** `NVIDIALakehouse` (schema-enabled) in workspace `NVIDIA-BalanceSheet-AI` before Run all.
**Requires:** gold statement tables from `nb_bronze_silver_gold_sec_edgar` (used to derive the start year).

## NVIDIA fiscal rule

- Fiscal year **ends on the last Sunday of January**.
- Fiscal year number / label = the **calendar year of that January end** (FY2026 ended 2026-01-25).
- FY start = day after prior FY end.
- **Q1–Q3** = 13 weeks each from FY start; **Q4** = remainder (13 or 14 weeks on 53-week years).
- Validated 24/24 against SEC submissions `reportDate` for 10-K/10-Q (FY2021–Q2 FY2027), including known checks FY2026→2026-01-25 and Q2 FY2027→2026-07-26.

Range: Jan 1 of the earliest calendar year in gold `period_end` values → end of the NVIDIA fiscal year that contains (~today + 2 years). Deterministic overwrite.


In [ ]:
# Imports + Spark write prefs (gold-friendly)
from __future__ import annotations

from datetime import date, timedelta
from calendar import monthrange

from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    DateType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

spark.conf.set("spark.sql.parquet.vorder.default", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.binSize", "1g")

TODAY = date.today()
print("today:", TODAY)


In [ ]:
# Fiscal helpers (last Sunday of January; Q1–Q3 = 13 weeks; Q4 = remainder)

def last_sunday_of_january(year: int) -> date:
    last = date(year, 1, 31)
    return last - timedelta(days=(last.weekday() + 1) % 7)  # Mon=0..Sun=6


def fiscal_year_end(fy: int) -> date:
    return last_sunday_of_january(fy)


def fiscal_year_start(fy: int) -> date:
    return fiscal_year_end(fy - 1) + timedelta(days=1)


def fiscal_quarter_bounds(fy: int) -> list[tuple[int, date, date]]:
    start = fiscal_year_start(fy)
    q1_end = start + timedelta(days=13 * 7 - 1)
    q2_end = q1_end + timedelta(days=13 * 7)
    q3_end = q2_end + timedelta(days=13 * 7)
    q4_end = fiscal_year_end(fy)
    return [
        (1, start, q1_end),
        (2, q1_end + timedelta(days=1), q2_end),
        (3, q2_end + timedelta(days=1), q3_end),
        (4, q3_end + timedelta(days=1), q4_end),
    ]


def classify_fiscal(d: date) -> tuple[int, int, date, date, date, date]:
    for fy in (d.year, d.year + 1):
        ys, ye = fiscal_year_start(fy), fiscal_year_end(fy)
        if ys <= d <= ye:
            for q, qs, qe in fiscal_quarter_bounds(fy):
                if qs <= d <= qe:
                    return fy, q, ys, ye, qs, qe
    raise ValueError(f"date out of fiscal map: {d}")


# Sanity: known SEC period ends
assert classify_fiscal(date(2026, 1, 25))[0:2] == (2026, 4)
assert classify_fiscal(date(2026, 7, 26))[0:2] == (2027, 2)
print("fiscal helpers ok")


In [ ]:
# Date range from gold facts → ~today+2y fiscal year end; build rows

def min_period_year() -> int:
    years = []
    for t in ("gold.balance_sheet", "gold.income_statement", "gold.cash_flow"):
        try:
            row = spark.sql(f"SELECT MIN(period_end) AS mn FROM {t}").collect()[0]
            if row["mn"] is not None:
                years.append(int(str(row["mn"])[:4]))
        except Exception as ex:
            print(f"warn: could not read {t}: {ex}")
    if not years:
        print("fallback start year 2007 (gold tables missing)")
        return 2007
    return min(years)


def range_end(today: date) -> date:
    horizon = today + timedelta(days=730)  # ~2 years
    fy, _, _, ye, _, _ = classify_fiscal(horizon)
    return ye


start = date(min_period_year(), 1, 1)
end = range_end(TODAY)
print(f"dim_date range: {start} → {end}")

MONTH_NAMES = [
    "", "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]
MONTH_SHORT = ["", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
DAY_NAMES = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]  # weekday() 0=Mon


def build_rows(start: date, end: date) -> list[dict]:
    rows = []
    d = start
    one = timedelta(days=1)
    while d <= end:
        fy, fq, fys, fye, fqs, fqe = classify_fiscal(d)
        iso_year, iso_week, iso_dow = d.isocalendar()  # dow Mon=1..Sun=7
        month_end_day = monthrange(d.year, d.month)[1]
        rows.append(
            {
                "date": d,
                "date_key": d.year * 10000 + d.month * 100 + d.day,
                "year": d.year,
                "quarter": (d.month - 1) // 3 + 1,
                "quarter_label": f"CY{d.year}Q{(d.month - 1) // 3 + 1}",
                "month": d.month,
                "month_name": MONTH_NAMES[d.month],
                "month_short": MONTH_SHORT[d.month],
                "year_month": f"{d.year:04d}-{d.month:02d}",
                "week_of_year": iso_week,
                "day_of_month": d.day,
                "day_of_week": iso_dow,
                "day_name": DAY_NAMES[d.weekday()],
                "is_weekend": d.weekday() >= 5,
                "month_start": date(d.year, d.month, 1),
                "month_end": date(d.year, d.month, month_end_day),
                "fiscal_year": fy,
                "fiscal_year_label": f"FY{fy}",
                "fiscal_quarter": fq,
                "fiscal_quarter_label": f"Q{fq} FY{fy}",
                "fiscal_year_start": fys,
                "fiscal_year_end": fye,
                "fiscal_quarter_start": fqs,
                "fiscal_quarter_end": fqe,
                "fiscal_week": ((d - fys).days // 7) + 1,
                "day_of_fiscal_year": (d - fys).days + 1,
                "is_fiscal_quarter_end": d == fqe,
                "is_fiscal_year_end": d == fye,
                "fiscal_year_quarter_sort": fy * 10 + fq,
            }
        )
        d += one
    return rows


rows = build_rows(start, end)
print(f"rows: {len(rows):,} | contiguous days check: {(end - start).days + 1 == len(rows)}")


In [ ]:
# Write gold.dim_date (overwrite + overwriteSchema)

SCHEMA = StructType(
    [
        StructField("date", DateType(), False),
        StructField("date_key", IntegerType(), False),
        StructField("year", IntegerType(), False),
        StructField("quarter", IntegerType(), False),
        StructField("quarter_label", StringType(), False),
        StructField("month", IntegerType(), False),
        StructField("month_name", StringType(), False),
        StructField("month_short", StringType(), False),
        StructField("year_month", StringType(), False),
        StructField("week_of_year", IntegerType(), False),
        StructField("day_of_month", IntegerType(), False),
        StructField("day_of_week", IntegerType(), False),
        StructField("day_name", StringType(), False),
        StructField("is_weekend", BooleanType(), False),
        StructField("month_start", DateType(), False),
        StructField("month_end", DateType(), False),
        StructField("fiscal_year", IntegerType(), False),
        StructField("fiscal_year_label", StringType(), False),
        StructField("fiscal_quarter", IntegerType(), False),
        StructField("fiscal_quarter_label", StringType(), False),
        StructField("fiscal_year_start", DateType(), False),
        StructField("fiscal_year_end", DateType(), False),
        StructField("fiscal_quarter_start", DateType(), False),
        StructField("fiscal_quarter_end", DateType(), False),
        StructField("fiscal_week", IntegerType(), False),
        StructField("day_of_fiscal_year", IntegerType(), False),
        StructField("is_fiscal_quarter_end", BooleanType(), False),
        StructField("is_fiscal_year_end", BooleanType(), False),
        StructField("fiscal_year_quarter_sort", IntegerType(), False),
    ]
)

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
sdf = spark.createDataFrame(rows, SCHEMA)
(
    sdf.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_date")
)
n = spark.table("gold.dim_date").count()
print(f"gold.dim_date: {n:,} rows")


In [ ]:
# Quality checks: uniqueness, contiguity, fiscal quarter-ends vs gold primary period_ends

dim = spark.table("gold.dim_date")
n = dim.count()
n_key = dim.select("date_key").distinct().count()
n_date = dim.select("date").distinct().count()
mn, mx = dim.select(F.min("date"), F.max("date")).collect()[0]
assert n == n_key == n_date, "date / date_key must be unique"
assert (mx.toordinal() - mn.toordinal() + 1) == n, "date range must be contiguous"
print(f"unique+contiguous ok | rows={n:,} | min={mn} max={mx}")

period_ends = spark.sql(
    """
    SELECT DISTINCT period_end
    FROM (
      SELECT period_end FROM gold.balance_sheet WHERE is_period_primary
      UNION
      SELECT period_end FROM gold.income_statement WHERE is_period_primary
      UNION
      SELECT period_end FROM gold.cash_flow WHERE is_period_primary
    )
    WHERE period_end IS NOT NULL
    """
)
chk = (
    period_ends.alias("p")
    .join(dim.alias("d"), F.col("p.period_end").cast("date") == F.col("d.date"), "left")
    .select(
        F.col("p.period_end"),
        F.col("d.date"),
        F.col("d.fiscal_quarter_label"),
        F.col("d.is_fiscal_quarter_end"),
    )
)
miss = chk.filter(F.col("date").isNull() | (F.col("is_fiscal_quarter_end") == F.lit(False)))
miss_n = miss.count()
total = chk.count()
print(f"primary period_end → is_fiscal_quarter_end: {total - miss_n}/{total}")
if miss_n:
    display(miss.orderBy("period_end"))
    raise AssertionError(f"{miss_n} primary period_end(s) not fiscal quarter ends")
display(chk.orderBy(F.col("period_end").desc()).limit(12))

print("\nSample fiscal quarter ends (recent):")
display(
    dim.filter(F.col("is_fiscal_quarter_end"))
    .select(
        "date",
        "fiscal_year_label",
        "fiscal_quarter_label",
        "fiscal_year_start",
        "fiscal_year_end",
        "fiscal_quarter_start",
        "fiscal_quarter_end",
        "fiscal_year_quarter_sort",
    )
    .orderBy(F.col("date").desc())
    .limit(16)
)

print("\nRow count / range:")
print(f"  gold.dim_date: {n:,} rows | {mn} → {mx}")
print(f"  fiscal quarter-end rows: {dim.filter(F.col('is_fiscal_quarter_end')).count():,}")
print(f"  fiscal year-end rows: {dim.filter(F.col('is_fiscal_year_end')).count():,}")
